# BanglaFinGPT — GPU run (Kaggle T4 / RTX 4050)

This notebook produces the numbers the CPU reproduction notebook could not: the
QLoRA fine-tuned scores, the ablation, the per-domain breakdown, the significance
tests, and — if you supply API keys — the hosted baselines with and without retrieval.

It writes `gpu_results.json` and `tables.tex`. The `.tex` file contains the finished
tables; paste them into the manuscript and the red `\needsnum{}` markers go away.

**Before you run**
1. Kaggle: Settings → Accelerator → **GPU T4 x2** (one is used), and Internet → **On**.
2. Add the dataset: either attach it as a Kaggle Dataset, or upload
   `BanglaFinGPT_dataset.xlsx` and set `DATA_PATH` below.
3. Runtime: fine-tuning 7,412 pairs for ~2 epochs is roughly 2.5–4 h on a T4 and
   1.5–2.5 h on an RTX 4050. Kaggle sessions cap at 12 h, so this fits, but set
   `MAX_STEPS` lower for a first smoke run.

**Precision note.** A T4 is Turing and has no bfloat16. The code resolves the dtype from
the device: fp16 on T4, bf16 on the RTX 4050. Do not hard-code bf16 on Kaggle — it fails
at model load.

## 0. Setup

In [ ]:
!pip -q install -U "transformers>=4.44" "peft>=0.12" "bitsandbytes>=0.43" \
    "accelerate>=0.33" "datasets>=2.20" "sentence-transformers>=3.0" \
    "openpyxl>=3.1" "sacrebleu" 2>&1 | tail -2

In [ ]:
import os, sys, json, time, random, subprocess
from pathlib import Path

REPO = "https://github.com/isratjahan829/Agent-Agnostic-Bilingual-AI-Chatbot-for-Financial-Literacy-in-Low-Resource-Languages"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

# The pipeline (splits, retrieval, prompts, filter, metrics) comes from the repo, so the
# GPU run and the CPU notebook use the same code rather than a re-implementation.
SRC = WORK / "banglafingpt-src"
if not SRC.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(SRC)], check=True)
sys.path.insert(0, str(SRC / "src"))

import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability()
    print(f"GPU: {name}  (compute capability {cap[0]}.{cap[1]})")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("bfloat16 supported:", cap[0] >= 8, "-> using", "bf16" if cap[0] >= 8 else "fp16")
else:
    raise SystemExit("No GPU. Kaggle: Settings -> Accelerator -> GPU T4.")

In [ ]:
from banglafingpt.config import load_config
from banglafingpt.utils import set_seed

SEED = 42
set_seed(SEED)

cfg = load_config(SRC / "configs" / "default.yaml")

# --- the only knobs you normally touch -----------------------------------------
DATA_PATH   = SRC / "data" / "BanglaFinGPT_dataset.xlsx"   # or your Kaggle input path
MAX_STEPS   = 2000          # ~2 epochs at an effective batch of 32; lower for a smoke run
EVAL_LIMIT  = None          # set e.g. 300 for a fast pass; None = the full test split
RUN_BASELINES = False       # needs API keys, see Section 6
# --------------------------------------------------------------------------------

cfg.train.output_dir = str(WORK / "adapters")
cfg.train.max_steps = MAX_STEPS
cfg.qlora.base_model = "BanglaLLM/bangla-llama-3.2-3b-instruct"

# A T4 holds the 3B model in 4-bit comfortably at batch 2 with more accumulation.
cfg.train.per_device_train_batch_size = 2
cfg.train.gradient_accumulation_steps = 16     # effective batch stays 32

print("dataset :", DATA_PATH, "|", "found" if Path(DATA_PATH).exists() else "MISSING")
print("adapters:", cfg.train.output_dir)
print("steps   :", cfg.train.max_steps)

## 1. Data, splits and retrieval index

In [ ]:
from banglafingpt.data.build_dataset import describe, leakage_report, split_pairs
from banglafingpt.data.load_xlsx import load_corpus

pairs, segments = load_corpus(DATA_PATH, chunk_words=220, overlap_words=40)
splits = split_pairs(pairs, 7412, 1000, 2000, seed=SEED)
train_pairs, val_pairs, test_pairs = splits["train"], splits["validation"], splits["test"]
if EVAL_LIMIT:
    test_pairs = test_pairs[:EVAL_LIMIT]

print(f"{len(pairs)} QA pairs over {len({s.doc_id for s in segments})} passages, "
      f"{len(segments)} chunks")
print({k: len(v) for k, v in splits.items()})
print("leakage:", leakage_report(splits))

In [ ]:
from banglafingpt.retrieval.embedder import load_embedder
from banglafingpt.retrieval.retriever import HybridRetriever

# On Kaggle with internet on, this is the real multilingual encoder the paper specifies.
embedder = load_embedder(cfg.retrieval.embedding_model)
print("encoder:", type(embedder).__name__, "dim", getattr(embedder, "dim", "?"))

t0 = time.time()
retriever = HybridRetriever.from_segments(segments, cfg.retrieval, embedder=embedder)
print(f"indexed {len(retriever.index)} chunks in {time.time() - t0:.0f}s")

In [ ]:
# Tune the dense/sparse mix on validation. With a trained encoder the dense half should
# carry more weight than it does in the CPU notebook's fallback configuration.
def recall_at_k(r, ps, k=5):
    hits = sum(p.source in {c.doc_id for c in r.retrieve(p.question, top_k=k)} for p in ps)
    return round(hits / len(ps), 3)

tuning_pairs = random.Random(SEED).sample(val_pairs, min(400, len(val_pairs)))
sweep = {}
for alpha in [0.0, 0.2, 0.3, 0.5, 0.7, 0.9, 1.0]:
    retriever.config.hybrid_alpha = alpha
    sweep[alpha] = recall_at_k(retriever, tuning_pairs)
    print(f"  alpha={alpha}  recall@5={sweep[alpha]}")

best_alpha = max(sweep, key=sweep.get)
retriever.config.hybrid_alpha = cfg.retrieval.hybrid_alpha = best_alpha
print(f"\nselected alpha = {best_alpha} (dense-only {sweep[1.0]}, BM25-only {sweep[0.0]})")

## 2. QLoRA fine-tuning

4-bit NF4 with double quantisation, LoRA rank 16 on all attention and MLP projections,
AdamW with cosine decay from 5e-5, loss on answer tokens only, and retrieved context in
the training prompts so they match what the model sees at inference.

In [ ]:
from banglafingpt.models.train import train

t0 = time.time()
adapter_path = train(cfg, train_pairs, val_pairs, retriever=retriever)
train_minutes = (time.time() - t0) / 60
print(f"\nadapters -> {adapter_path}")
print(f"wall-clock: {train_minutes:.0f} minutes on {torch.cuda.get_device_name(0)}")

## 3. Evaluation — the ablation of Tables 5 and 7

In [ ]:
from banglafingpt.eval.evaluate import evaluate, per_example_records, run_system
from banglafingpt.hallucination.filters import HallucinationFilter
from banglafingpt.pipeline import BanglaFinGPT

cfg.agent.backend = "local"

def build(adapter, use_retrieval, use_filter):
    cfg.agent.adapter_path = adapter
    system = BanglaFinGPT(
        cfg, retriever=retriever,
        hallucination_filter=HallucinationFilter(cfg.hallucination, embedder=embedder),
        use_retrieval=use_retrieval, use_filter=use_filter)
    return system

VARIANTS = [
    ("base",         None,         False, False),
    ("ft",           adapter_path, False, False),
    ("rag",          None,         True,  False),
    ("ft_rag",       adapter_path, True,  False),
    ("banglafingpt", adapter_path, True,  True),
]

reports, records, em_scores = {}, {}, {}
for name, adapter, use_r, use_f in VARIANTS:
    t0 = time.time()
    answers = run_system(build(adapter, use_r, use_f), test_pairs, progress_every=200)
    reports[name] = evaluate(test_pairs, answers)
    records[name] = per_example_records(test_pairs, answers)
    em_scores[name] = [float(r["scores"]["em"]) for r in records[name]]
    o = reports[name]["overall"]
    print(f"{name:<14} EM={o['em_pct']:>5}%  F1={o['f1']:.3f}  "
          f"refused={reports[name]['refusal_rate_pct']}%  ({time.time() - t0:.0f}s)")

In [ ]:
import pandas as pd

ablation = pd.DataFrame([
    {"configuration": n,
     "EM (%)": reports[n]["overall"]["em_pct"], "F1": reports[n]["overall"]["f1"],
     "BLEU-4": reports[n]["overall"]["bleu4"], "ROUGE-L": reports[n]["overall"]["rouge_l"],
     "METEOR": reports[n]["overall"]["meteor"],
     "refused %": reports[n]["refusal_rate_pct"]}
    for n, *_ in VARIANTS]).set_index("configuration")
ablation

## 4. Per-domain breakdown and significance

In [ ]:
from banglafingpt.eval.significance import significance_table

by_domain = pd.DataFrame(reports["banglafingpt"]["by_domain"]).T[
    ["n", "em_pct", "f1"]].sort_values("f1", ascending=False)
print(by_domain.to_string(), "\n")

sig = pd.DataFrame(significance_table(em_scores, baseline="base", metric="em"))
print(sig.to_string(index=False))

## 5. Hallucination audit

In [ ]:
from banglafingpt.eval.annotation import build_annotation_sheets, reduction_report
from banglafingpt.eval.error_analysis import error_report, hallucination_report

audit = pd.DataFrame([{"variant": n, **hallucination_report(records[n])}
                      for n, *_ in VARIANTS]).set_index("variant")
print(audit.to_string(), "\n")

before = audit.loc["ft_rag", "hallucinated_pct"]
after = audit.loc["banglafingpt", "hallucinated_pct"]
framing = reduction_report(before, after)
print("filter effect:", framing)

errors = error_report(records["banglafingpt"])
print("\nerror types:", {r["error_type"]: r["pct"] for r in errors["by_type"]})

# Blind sheets for the three annotators (Section 3.6.1 of the manuscript).
sheets = build_annotation_sheets(records["banglafingpt"], WORK / "annotation",
                                 annotators=("A1", "A2", "A3"), sample_size=200, seed=SEED)
print("\nannotation sheets:", sheets["sheets"])

## 6. Hosted baselines, with and without the same retrieved context

This is the fairness fix Reviewer 1 asked for. It needs API keys — add them as Kaggle
Secrets (Add-ons → Secrets) and set `RUN_BASELINES = True` in Section 0.

In [ ]:
baseline_rows = []
if RUN_BASELINES:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for env, secret in [("OPENAI_API_KEY", "OPENAI_API_KEY"),
                        ("ANTHROPIC_API_KEY", "ANTHROPIC_API_KEY"),
                        ("GOOGLE_API_KEY", "GOOGLE_API_KEY")]:
        try:
            os.environ[env] = secrets.get_secret(secret)
        except Exception as exc:
            print(f"  {env}: not set ({exc})")

    for backend, model in [("openai", "gpt-4o"), ("anthropic", "claude-sonnet-4-5"),
                           ("gemini", "gemini-1.5-pro")]:
        if not os.environ.get({"openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY",
                               "gemini": "GOOGLE_API_KEY"}[backend]):
            continue
        cfg.agent.backend, cfg.agent.model, cfg.agent.adapter_path = backend, model, None
        for label, use_r in [("no retrieval", False), ("+ RAG context", True)]:
            system = BanglaFinGPT(cfg, retriever=retriever, use_retrieval=use_r,
                                  use_filter=False)
            answers = run_system(system, test_pairs, progress_every=200)
            o = evaluate(test_pairs, answers)["overall"]
            baseline_rows.append({"model": model, "setting": label, **o})
            print(f"  {model} {label}: EM={o['em_pct']}%  F1={o['f1']}")
    cfg.agent.backend = "local"
else:
    print("RUN_BASELINES is False - skipping. Table 5's '+ RAG context' rows stay empty.")

pd.DataFrame(baseline_rows) if baseline_rows else None

## 7. Write the results and the finished LaTeX tables

In [ ]:
from banglafingpt.utils import write_json, write_jsonl

results = {
    "environment": {
        "gpu": torch.cuda.get_device_name(0),
        "compute_capability": list(torch.cuda.get_device_capability()),
        "precision": "bf16" if torch.cuda.get_device_capability()[0] >= 8 else "fp16",
        "train_minutes": round(train_minutes, 1),
        "encoder": type(embedder).__name__,
        "embedding_model": cfg.retrieval.embedding_model,
        "hybrid_alpha": best_alpha,
        "max_steps": cfg.train.max_steps,
        "test_size": len(test_pairs),
    },
    "retrieval_sweep": {str(k): v for k, v in sweep.items()},
    "ablation": ablation.reset_index().to_dict("records"),
    "by_domain": reports["banglafingpt"]["by_domain"],
    "significance": sig.to_dict("records"),
    "hallucination": audit.reset_index().to_dict("records"),
    "hallucination_framing": framing,
    "errors": errors,
    "baselines": baseline_rows,
}
write_json(WORK / "gpu_results.json", results)
for name in records:
    write_jsonl(WORK / f"predictions_{name}.jsonl", records[name])
print("wrote", WORK / "gpu_results.json")

In [ ]:
# Render the manuscript tables with the measured values.
def fmt(x, nd=2):
    return f"{x:.{nd}f}"

lines = []
lines.append("% ---- Table 5 / Table 7: ablation ----")
for row in ablation.reset_index().to_dict("records"):
    lines.append(f"{row['configuration']} & {fmt(row['EM (%)'], 1)} & {fmt(row['F1'])} & "
                 f"{fmt(row['BLEU-4'])} & {fmt(row['ROUGE-L'])} & {fmt(row['METEOR'])} \\\\")

if baseline_rows:
    lines.append("")
    lines.append("% ---- Table 5: hosted baselines ----")
    for row in baseline_rows:
        lines.append(f"{row['model']} ({row['setting']}) & {fmt(row['em_pct'], 1)} & "
                     f"{fmt(row['f1'])} & {fmt(row['bleu4'])} & {fmt(row['rouge_l'])} & "
                     f"{fmt(row['meteor'])} \\\\")

lines.append("")
lines.append("% ---- Table 8: per domain ----")
for domain, row in reports["banglafingpt"]["by_domain"].items():
    lines.append(f"{domain} & {int(row['n'])} & {fmt(row['em_pct'], 1)} & {fmt(row['f1'])} \\\\")

lines.append("")
lines.append("% ---- Table 12: significance ----")
for row in sig.to_dict("records"):
    p = "---" if row["p_value"] is None else (
        "$p < 0.001$" if row["p_value"] < 0.001 else f"$p = {row['p_value']:.3f}$")
    lines.append(f"{row['system']} & {fmt(row['mean'], 1)} & "
                 f"[{row['ci95'][0]}, {row['ci95'][1]}] & {p} \\\\")

lines.append("")
lines.append("% ---- Section 2.4: hardware sentence ----")
lines.append(f"% trained on {torch.cuda.get_device_name(0)} in "
             f"{train_minutes:.0f} minutes, "
             f"{'bf16' if torch.cuda.get_device_capability()[0] >= 8 else 'fp16'}")

lines.append("")
lines.append("% ---- Section 3.6: filter effect ----")
lines.append(f"% {framing['before_pct']}\\% -> {framing['after_pct']}\\%, "
             f"{framing['absolute_reduction_pp']} pp absolute, "
             f"{framing['relative_reduction_pct']}\\% relative, "
             f"one per {framing['number_needed_to_filter']} queries")

(WORK / "tables.tex").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines[:20]))
print("\n...\nwrote", WORK / "tables.tex")

## 8. What to do with the output

1. Download `tables.tex` and paste each block into the matching table in `main.tex`.
2. Download `gpu_results.json` and keep it with the submission — it is the record of
   which configuration produced the numbers.
3. Fill the Section 2.4 hardware sentence from the comment at the bottom of `tables.tex`.
4. Hand the three sheets in `annotation/` to your annotators, then score them:

```python
from banglafingpt.eval.annotation import load_annotations, score_annotations
score_annotations(load_annotations(sheets["sheets"]))   # -> rates and Fleiss' kappa
```

5. Re-check that no `\needsnum{}` marker is left:

```bash
grep -o 'needsnum{' main.tex | wc -l    # must be 0
```